# Part E: three lightweight steps, one small model each

Parts C and D showed that changing only the output (soft targets, the span-F1 loss) gives nothing
once the cut-off is tuned. What helped was a model that knows more: averaging 6 models gave 57.34.
Each step below keeps the final detector at **one ModernBERT-base model (0.15B)**.

| Step | What it does | GPU time on a T4 |
|---|---|---|
| 1 | Average the **weights** of 4 models we already trained into one model (no training) | about 20-30 min |
| 2 | Same training as `mb_hard_s43`, but start from **ModernBERT-base-nli**, a copy already trained to judge "is this supported?" | about 80-90 min |
| 3 | Same training as `mb_hard_s43`, plus about 2,900 **extra summaries with known errors** made from error-free training summaries | about 90-100 min |

Numbers to compare with: one of our models after tuning, **about 55.2**; the 6-model average,
**57.34**; LettuceDetect-large (0.4B), **58.93**; RL4HS-14B, **58.3**; RLSeek-14B, **60.1**.

Rules: the candidates are fixed here, before seeing test results. Cut-offs are fitted on dev only.
Steps 2 and 3 use seed 43 so they can be compared with `mb_hard_s43`, which differs only in the
one thing being tested.

Select **Runtime > Change runtime type > T4 GPU**, then run the cells in order. Every step saves its
work to Drive; if a cell stops (time limit or disconnection), run it again and it continues.
You can run the steps on different days.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, time, importlib.util
from pathlib import Path
from IPython.display import Markdown, display

REPO = Path('/content/tse')
BRANCH = 'fix/runnable-pipeline'
def setup_command(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    result.check_returncode()

if not REPO.exists():
    setup_command(['git', 'clone', '--single-branch', '--branch', BRANCH,
                   'https://github.com/gbirhanu/tse.git', str(REPO)])
else:
    actual = subprocess.check_output(['git', '-C', str(REPO), 'branch', '--show-current'], text=True).strip()
    assert actual == BRANCH, f'Expected {BRANCH}, found {actual}'
    setup_command(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH])
PROJECT = REPO / 'hallu_poc'
WORK = Path('/content/drive/MyDrive/hallu_poc/work')
DATA = Path('/content/drive/MyDrive/hallu_poc/RAGTruth')
if not (DATA / 'response.jsonl').exists() and not (DATA / 'dataset/response.jsonl').exists():
    DATA = Path('/content/RAGTruth')
    if not DATA.exists():
        setup_command(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)])
setup_command([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'tqdm>=4.27,<5'])
REVISION = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Code revision:', REVISION)
print('Saved outputs:', WORK)

## Helper and the saved models this notebook needs

This cell checks that the Part C models and predictions are on Drive and shows their results.
It does not train anything.

In [ ]:
sys.path.insert(0, str(PROJECT))
import importlib
import hallu.notebook
importlib.reload(hallu.notebook)
from hallu.notebook import run_command

def run(*arguments, timeout=110 * 60):
    command = [sys.executable, '-u', str(PROJECT / 'poc.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK)]
    environment = {**os.environ, 'TOKENIZERS_PARALLELISM': 'false'}
    try:
        if arguments[0] == 'report':
            result = subprocess.run(command, cwd=PROJECT, timeout=timeout, env=environment,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            if result.returncode:
                print(result.stdout, flush=True)
                result.check_returncode()
            display(Markdown(result.stdout))
            return result.stdout
        result = run_command(command, cwd=PROJECT, env=environment, timeout=timeout)
        table = '\n'.join(line for line in result.stdout.splitlines() if line.startswith('|'))
        if table:
            display(Markdown(table))
        return result.stdout
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError('Time budget reached. Work is saved on Drive; run this cell again to continue.') from exc

def gpu_deadline(minutes=110):
    import torch
    assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU first.'
    print('GPU:', torch.cuda.get_device_name(0), flush=True)
    deadline = time.monotonic() + minutes * 60
    def remaining():
        seconds = deadline - time.monotonic()
        if seconds <= 0:
            raise RuntimeError(f'{minutes}-minute budget reached. Run this cell again to continue.')
        return seconds
    return remaining

help_text = subprocess.check_output([sys.executable, str(PROJECT / 'poc.py'), '--help'], text=True)
assert 'soup' in help_text and '--extra-train' in help_text, 'Push the Part E commit, then rerun setup.'
SEEDS = ['mb_hard_s43', 'mb_hard_s44', 'mb_soft_s43', 'mb_soft_s44']
required = ['preds/ld_large_spans.pkl', 'results/ld_large_levers.json']
for name in SEEDS:
    required += [f'models/{name}/best/model.safetensors', f'models/{name}/best/hallu_run.json',
                 f'preds/{name}_dev.pkl', f'preds/{name}_test.pkl', f'results/{name}_levers.json']
missing = [name for name in required if not (WORK / name).exists()]
assert not missing, f'Missing saved files: {missing}. Use the work folder from Parts A-D.'
REFERENCES = ['ld_large', 'ens_hard_soft_6seed'] + SEEDS
run('report', '--runs', *REFERENCES)

## Step 1: average the weights of four models (no training)

`soup_hs4` is one model whose weights are the plain average of `mb_hard_s43`, `mb_hard_s44`,
`mb_soft_s43` and `mb_soft_s44`. All four started from the same ModernBERT-base, so their weights
may average well. `ens_hs4` averages the same four models' **outputs** instead, which needs four
models at run time; it shows the most the weight average could reach.

**Step 1 worked if** `soup_hs4` (tuned) is clearly above the single models (about 55) and close to
`ens_hs4`. Then we have the ensemble's gain in a single 0.15B model.

In [ ]:
remaining = gpu_deadline(110)
run('soup', '--name', 'soup_hs4', '--members', *SEEDS, '--device', 'cuda', timeout=remaining())
run('levers', '--name', 'soup_hs4', '--fit', 'dev', '--against', 'ld_large:baseline', timeout=remaining())
run('ensemble', '--name', 'ens_hs4', '--members', *SEEDS, '--fit', 'dev',
    '--against', 'ld_large:baseline', timeout=remaining())
run('report', '--runs', 'soup_hs4', 'ens_hs4', *SEEDS, 'ld_large',
    '--compare', 'ens_hs4', 'soup_hs4', timeout=10 * 60)

## Step 2: start from a model that already judges "supported or not"

`nli_hard_s43` is trained exactly like `mb_hard_s43` (hard labels, seed 43, 3 epochs, learning rate
2e-5, 4,096 tokens), but starts from
[tasksource/ModernBERT-base-nli](https://huggingface.co/tasksource/ModernBERT-base-nli), the same
0.15B model already trained on natural-language-inference data. Only its last 2-class layer is new.

**Step 2 worked if** `nli_hard_s43` (tuned) beats `mb_hard_s43` (55.26), with the paired interval
below the table above zero, or clearly improves the Summary column.

In [ ]:
remaining = gpu_deadline(110)
run('train', '--name', 'nli_hard_s43', '--base', 'tasksource/ModernBERT-base-nli',
    '--targets', 'hard', '--seed', '43', '--epochs', '3', '--lr', '2e-5',
    '--max-length', '4096', '--batch-tokens', '8192', '--checkpoint-every', '200',
    '--device', 'cuda', timeout=remaining())
run('levers', '--name', 'nli_hard_s43', '--fit', 'dev', '--against', 'ld_large:baseline', timeout=remaining())
run('report', '--runs', 'nli_hard_s43', 'mb_hard_s43', 'ld_large',
    '--compare', 'mb_hard_s43', 'nli_hard_s43', timeout=10 * 60)

## Step 3: extra training summaries with known errors

Summaries are our weakest task (about 37 against LettuceDetect's 44.6) and have the fewest marked
errors in the training data. `synth` takes each error-free **training** summary (never dev or test)
and makes one copy with a known error:

- **swap**: one number, year, month, weekday or name is changed so it no longer matches the article
  (a replacement name or number is often taken from the same article);
- **move**: a sentence that annotators marked as hallucinated in another model's summary of the same
  article is inserted at a sentence break.

The changed words are the marked span. Names are found with spaCy's small English model. The cell
prints a few examples; read them. Then `mb_hard_syn_s43` is trained exactly like `mb_hard_s43`,
plus these extra summaries.

**Step 3 worked if** `mb_hard_syn_s43` (tuned) beats `mb_hard_s43` (55.26), and above all if its
Summary column rises from about 37.

In [ ]:
if importlib.util.find_spec('spacy') is None:
    setup_command([sys.executable, '-m', 'pip', 'install', '-q', 'spacy>=3.7,<4'])
if importlib.util.find_spec('en_core_web_sm') is None:
    setup_command([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
run('synth', '--name', 'syn_sum_v1', timeout=30 * 60)
remaining = gpu_deadline(110)
run('train', '--name', 'mb_hard_syn_s43', '--base', 'answerdotai/ModernBERT-base',
    '--targets', 'hard', '--seed', '43', '--epochs', '3', '--lr', '2e-5',
    '--max-length', '4096', '--batch-tokens', '8192', '--checkpoint-every', '200',
    '--extra-train', 'syn_sum_v1', '--device', 'cuda', timeout=remaining())
run('levers', '--name', 'mb_hard_syn_s43', '--fit', 'dev', '--against', 'ld_large:baseline', timeout=remaining())
run('report', '--runs', 'mb_hard_syn_s43', 'mb_hard_s43', 'ld_large',
    '--compare', 'mb_hard_s43', 'mb_hard_syn_s43', timeout=10 * 60)

## Optional: steps 2 and 3 together

Run this **only if steps 2 and 3 both helped**. It starts from the NLI model and adds the extra
summaries. Otherwise skip to the last cell.

In [ ]:
RUN_COMBINED = False   # set to True only if steps 2 and 3 both helped
if RUN_COMBINED:
    remaining = gpu_deadline(110)
    run('train', '--name', 'nli_hard_syn_s43', '--base', 'tasksource/ModernBERT-base-nli',
        '--targets', 'hard', '--seed', '43', '--epochs', '3', '--lr', '2e-5',
        '--max-length', '4096', '--batch-tokens', '8192', '--checkpoint-every', '200',
        '--extra-train', 'syn_sum_v1', '--device', 'cuda', timeout=remaining())
    run('levers', '--name', 'nli_hard_syn_s43', '--fit', 'dev', '--against', 'ld_large:baseline',
        timeout=remaining())
    run('report', '--runs', 'nli_hard_syn_s43', 'nli_hard_s43', 'mb_hard_syn_s43', 'mb_hard_s43',
        'ld_large', '--compare', 'mb_hard_s43', 'nli_hard_syn_s43', timeout=10 * 60)
else:
    print('Skipped. Set RUN_COMBINED = True only if steps 2 and 3 both helped.')

## Final table

Shows every Part E result next to the references and saves it to
`MyDrive/hallu_poc/work/results/part_e_report.txt`. No GPU is needed. Paste this table back into
the chat. If the runtime was reset, run the first two cells again first.

In [ ]:
runs = ['ld_large', 'ens_hard_soft_6seed', *SEEDS, 'soup_hs4', 'ens_hs4', 'nli_hard_s43',
        'mb_hard_syn_s43', 'nli_hard_syn_s43']
report = run('report', '--runs', *runs, timeout=10 * 60)
report_path = WORK / 'results' / 'part_e_report.txt'
report_path.write_text(report, encoding='utf-8')
print('Table saved to:', report_path)